# Week 6 — Three-Layer XAI

**Prerequisite:** `checkpoints/E2_quantum_best.pth` must exist.

| Layer | Method | Shows |
|-------|--------|-------|
| 1 | Grad-CAM | Which tissue regions drive prediction |
| 2 | GAT attention αᵢⱼ | Which patch connections matter |
| 3 | VQC parameter-shift + Bloch sphere | Quantum gate sensitivity |

Runs ~2-3 hrs on laptop (inference only, no retraining).

In [ ]:
import sys, json, gc
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

sys.path.insert(0, '..')
from pathq.model_v2   import QuantaPathV2
from pathq.dataset_v2 import get_loaders_from_features
import pennylane as qml

DEVICE   = torch.device('cuda')
ROOT     = Path('..')
FEAT_DIR = Path('./data/features_uni')
CKPT_DIR = ROOT / 'checkpoints'
OUT_DIR  = ROOT / 'outputs'
XAI_DIR  = ROOT / 'outputs' / 'xai'
XAI_DIR.mkdir(parents=True, exist_ok=True)

print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

ckpt_path = CKPT_DIR / 'E2_quantum_best.pth'
assert ckpt_path.exists(), \
    'E2_quantum_best.pth not found \u2014 run Week 5 E2 first'
print(f'Checkpoint: {ckpt_path} \u2705')

In [ ]:
model = QuantaPathV2(
    use_vqc    = True,
    n_qubits   = 3,
    vqc_layers = 2,
    in_dim     = 1040,
).to(DEVICE)

ck = torch.load(CKPT_DIR / 'E2_quantum_best.pth', weights_only=False)
model.load_state_dict(ck['model_state'])
model.eval()

print(f'Loaded E2_quantum_best.pth')
print(f'  Saved at epoch : {ck["epoch"]}')
print(f'  Best val AUC   : {ck["best_val_auc"]:.4f}')

_, val_loader, test_loader = get_loaders_from_features(
    features_dir = FEAT_DIR,
    batch_size   = 1,
    k            = 8,
    seed         = 42,
    max_patches  = 3000,
)

tumour_batch = None
normal_batch = None

for batch in test_loader:
    if batch.y.item() == 1 and tumour_batch is None:
        tumour_batch = batch
    if batch.y.item() == 0 and normal_batch is None:
        normal_batch = batch
    if tumour_batch is not None and normal_batch is not None:
        break

print(f'Tumour slide: {tumour_batch.num_nodes} patches')
print(f'Normal slide: {normal_batch.num_nodes} patches')

In [ ]:
# Layer 1: Which regions drive the prediction?
# Gradient of prediction w.r.t. node features at input_proj input

def gradcam_node_importance(model, batch, device):
    batch = batch.to(device)

    # Capture gradient at the input to input_proj
    captured = {}

    def hook_fn(module, inp, out):
        captured['x'] = inp[0]
        captured['x'].retain_grad()

    hook = model.input_proj.register_forward_hook(hook_fn)
    logits, _ = model(batch)
    pred_class = logits.argmax(dim=1).item()
    logits[0, pred_class].backward()
    hook.remove()

    grad       = captured['x'].grad          # (N, proj_in)
    importance = grad.abs().mean(dim=1)      # (N,)
    importance = importance.detach().cpu().numpy()
    importance = (importance - importance.min()) / \
                 (importance.max() - importance.min() + 1e-8)
    return importance, pred_class

print('Computing Layer 1 \u2014 Spatial XAI (Grad-CAM)...')

tumour_imp, tumour_pred = gradcam_node_importance(model, tumour_batch, DEVICE)
normal_imp, normal_pred = gradcam_node_importance(model, normal_batch, DEVICE)

tumour_coords = tumour_batch.coords.numpy()
normal_coords = normal_batch.coords.numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Layer 1 \u2014 Spatial XAI: Patch Importance (Grad-CAM)',
             fontsize=13, fontweight='bold')

for ax, coords, imp, label, pred in [
    (axes[0], tumour_coords, tumour_imp, 'Tumour slide', tumour_pred),
    (axes[1], normal_coords, normal_imp, 'Normal slide', normal_pred),
]:
    pred_str = 'Tumour' if pred == 1 else 'Normal'
    sc = ax.scatter(coords[:,0], coords[:,1], c=imp,
                    cmap='Reds', s=15, alpha=0.8, vmin=0, vmax=1)
    plt.colorbar(sc, ax=ax, label='Importance')
    ax.set_title(f'{label} \u2192 Predicted: {pred_str}', fontsize=11)
    ax.set_xlabel('Col position')
    ax.set_ylabel('Row position')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(XAI_DIR / 'layer1_spatial_gradcam.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u2705 Saved: outputs/xai/layer1_spatial_gradcam.png')

In [ ]:
# Layer 2: Which patch connections matter most?
# GAT attention weights — must pass 256-dim features (after input_proj)

def get_gat_attention(model, batch, device):
    batch = batch.to(device)
    with torch.no_grad():
        # Compute 256-dim features (input_proj output)
        uni = batch.x[:, :model.feat_dim]
        pe  = batch.x[:, model.feat_dim:]
        if model.use_vqc:
            x_in = torch.cat([model.vqc(uni), pe], dim=1)
        else:
            x_in = batch.x
        x_h = model.input_proj(x_in)   # (N, 256)

        # Call GAT with return_attention_weights=True
        _, (edge_index_out, attn) = model.gat_mamba.gat(
            x_h, batch.edge_index,
            edge_attr              = batch.edge_attr,
            return_attention_weights = True,
        )
    return edge_index_out, attn.mean(dim=1).detach().cpu().numpy()

print('Computing Layer 2 \u2014 Structural XAI (GAT Attention)...')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Layer 2 \u2014 Structural XAI: GAT Attention Weights \u03b1\u1d62\u2c7c',
             fontsize=13, fontweight='bold')

for ax, batch, title in [
    (axes[0], tumour_batch, 'Tumour slide'),
    (axes[1], normal_batch, 'Normal slide'),
]:
    edge_idx, attn = get_gat_attention(model, batch, DEVICE)
    src    = edge_idx[0].cpu().numpy()
    tgt    = edge_idx[1].cpu().numpy()
    coords = batch.coords.numpy()

    ax.scatter(coords[:,0], coords[:,1], c='lightgray', s=10, zorder=2)

    top_idx   = np.argsort(attn)[-100:]
    attn_norm = (attn - attn.min()) / (attn.max() - attn.min() + 1e-8)

    for idx in top_idx:
        s, t = src[idx], tgt[idx]
        ax.plot([coords[s,0], coords[t,0]],
                [coords[s,1], coords[t,1]],
                alpha=float(attn_norm[idx]),
                color='royalblue', linewidth=0.8)

    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Col position')
    ax.set_ylabel('Row position')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(XAI_DIR / 'layer2_structural_gat_attention.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u2705 Saved: outputs/xai/layer2_structural_gat_attention.png')

In [ ]:
# Layer 3a: Which VQC gates matter most?
# Gradient of tumour-class score w.r.t. VQC weight parameters

print('Computing Layer 3 \u2014 Quantum XAI (parameter gradients)...')

def get_vqc_gradients(model, batch, device):
    batch = batch.to(device)
    model.zero_grad()
    logits, _ = model(batch)
    logits[0, 1].backward()   # tumour class score
    grads = {}
    for name, param in model.named_parameters():
        if 'vqc' in name and param.grad is not None:
            grads[name] = param.grad.abs().detach().cpu().numpy()
    return grads

tumour_grads = get_vqc_gradients(model, tumour_batch, DEVICE)
normal_grads = get_vqc_gradients(model, normal_batch, DEVICE)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
fig.suptitle('Layer 3 \u2014 Quantum XAI: VQC Parameter Sensitivity\n'
             '(Gradient magnitude w.r.t. rotation angles)',
             fontsize=13, fontweight='bold')

for ax, grads, title, color in [
    (axes[0], tumour_grads, 'Tumour slide \u2014 gate sensitivities', '#C04040'),
    (axes[1], normal_grads, 'Normal slide \u2014 gate sensitivities', '#4060C0'),
]:
    names, values = [], []
    for k, v in grads.items():
        for i, val in enumerate(v.flatten()):
            short = k.split('.')[-1]
            names.append(f'{short}[{i}]')
            values.append(float(val))

    ax.bar(range(len(values)), values, color=color, alpha=0.8, width=0.7)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('|Gradient|')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('VQC parameter')

plt.tight_layout()
plt.savefig(XAI_DIR / 'layer3_quantum_gradients.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u2705 Saved: outputs/xai/layer3_quantum_gradients.png')

In [ ]:
# Layer 3b: Bloch sphere — quantum state per patch for tumour vs normal

print('Computing Bloch sphere trajectories...')

def get_bloch_vectors(model, batch, device, n_patches=5):
    batch = batch.to(device)
    with torch.no_grad():
        uni = batch.x[:n_patches, :model.feat_dim]   # (5, 1024)
        p   = model.vqc.proj(uni)                     # (5, 3) bounded by Tanh
        vecs = []
        for i in range(p.shape[0]):
            z = model.vqc.vqc(p[i]).detach().cpu().numpy()  # (3,) PauliZ per qubit
            vecs.append(z)
    return np.array(vecs)  # (5, 3)

tumour_bloch = get_bloch_vectors(model, tumour_batch, DEVICE)
normal_bloch = get_bloch_vectors(model, normal_batch, DEVICE)

fig = plt.figure(figsize=(14, 6))
fig.suptitle('Layer 3 \u2014 Quantum XAI: Bloch Sphere State Trajectories\n'
             'Tumour vs Normal patch quantum states (\u27e8Z\u27e9 per qubit)',
             fontsize=13, fontweight='bold')

for subplot_idx, (bloch, title, color) in enumerate([
    (tumour_bloch, 'Tumour patches', '#C04040'),
    (normal_bloch, 'Normal patches', '#4060C0'),
]):
    ax = fig.add_subplot(1, 2, subplot_idx+1, projection='3d')

    u = np.linspace(0, 2*np.pi, 30)
    v = np.linspace(0, np.pi, 20)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones_like(u), np.cos(v))
    ax.plot_wireframe(xs, ys, zs, color='lightgray', alpha=0.2, linewidth=0.5)

    n_p     = bloch.shape[0]
    cmap    = plt.cm.Reds if color == '#C04040' else plt.cm.Blues
    colors_p = cmap(np.linspace(0.4, 1.0, n_p))

    for i in range(n_p):
        z0 = float(bloch[i, 0])
        z1 = float(bloch[i, 1]) if bloch.shape[1] > 1 else 0.0
        z2 = float(bloch[i, 2]) if bloch.shape[1] > 2 else 0.0
        ax.quiver(0, 0, 0, z1, z2, z0,
                  color=colors_p[i], arrow_length_ratio=0.15,
                  linewidth=2, alpha=0.8)

    ax.set_xlim([-1.2, 1.2]); ax.set_ylim([-1.2, 1.2]); ax.set_zlim([-1.2, 1.2])
    ax.set_xlabel('\u27e8X\u27e9'); ax.set_ylabel('\u27e8Y\u27e9'); ax.set_zlabel('\u27e8Z\u27e9')
    ax.set_title(title, fontsize=11)

plt.tight_layout()
plt.savefig(XAI_DIR / 'layer3_bloch_sphere.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u2705 Saved: outputs/xai/layer3_bloch_sphere.png')

In [ ]:
# Combined paper figure (Figure 3)
from PIL import Image

fig = plt.figure(figsize=(18, 12))
fig.suptitle(
    'QuantaPath v2 \u2014 Three-Layer Explainable AI\n'
    'CAMELYON16 \u00b7 UNI (ViT-L) + Quantum VQC + GAT-Transformer',
    fontsize=14, fontweight='bold', y=0.98
)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

xai_files = [
    ('layer1_spatial_gradcam.png',         'Layer 1 \u2014 Spatial\nGrad-CAM on UNI'),
    ('layer2_structural_gat_attention.png','Layer 2 \u2014 Structural\nGAT Attention \u03b1\u1d62\u2c7c'),
    ('layer3_quantum_gradients.png',       'Layer 3 \u2014 Quantum\nParameter-shift gradients'),
    ('layer3_bloch_sphere.png',            'Layer 3 \u2014 Quantum\nBloch sphere trajectories'),
]

positions = [gs[0,0], gs[0,1], gs[0,2], gs[1,0:2]]

for pos, (fname, title) in zip(positions, xai_files):
    fpath = XAI_DIR / fname
    if fpath.exists():
        ax  = fig.add_subplot(pos)
        img = Image.open(fpath)
        ax.imshow(img)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.axis('off')

ax_sum = fig.add_subplot(gs[1, 2])
ax_sum.axis('off')
summary = (
    'XAI Summary\n\n'
    'Layer 1 (Spatial):\n'
    '  Grad-CAM on UNI backbone\n'
    '  Shows which tissue regions\n'
    '  drive the prediction\n\n'
    'Layer 2 (Structural):\n'
    '  GAT attention weights \u03b1\u1d62\u2c7c\n'
    '  Shows which patch connections\n'
    '  matter most\n\n'
    'Layer 3 (Quantum):\n'
    '  Parameter-shift gradients\n'
    '  Bloch sphere trajectories\n'
    '  Shows quantum gate sensitivity'
)
ax_sum.text(0.05, 0.95, summary,
            transform=ax_sum.transAxes,
            fontsize=9, verticalalignment='top',
            fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='#f8f8f8', alpha=0.8))

plt.savefig(OUT_DIR / 'week6_xai_report.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print('='*55)
print('WEEK 6 XAI COMPLETE')
print('='*55)
print('Saved: outputs/week6_xai_report.png  \u2190 Paper Figure 3')
print()
print('XAI files created:')
for f in sorted(XAI_DIR.glob('*.png')):
    print(f'  {f.name}')

In [ ]:
# Final results table — loads all available Week 5 + 6 JSONs

print('='*65)
print('QUANTAPATH v2 \u2014 COMPLETE RESULTS SUMMARY')
print('='*65)

exp_files = {
    'E1_classical':          'UNI + Classical GAT-Transformer',
    'E2_quantum':            'UNI + Quantum VQC+GAT (3q 2L)',
    'E3a_classical_20pct':   'UNI + Classical [20% data]',
    'E3b_quantum_20pct':     'UNI + Quantum VQC [20% data]',
    'E4_resnet':             'ResNet-50 + Classical GAT-Transformer',
}

def load_result(exp_id):
    fp = OUT_DIR / f'{exp_id}_result.json'
    if not fp.exists():
        return None
    r = json.load(open(fp))
    res = r.get('results', r)
    return {
        'val_auc': res.get('val_auc', 0),
        'test_auc': res.get('test_auc', res.get('auc', 0)),
        'f1':       res.get('f1', 0),
        'gap':      res.get('gap', 0),
    }

results = {}
print()
print('TABLE 1 \u2014 Full dataset (CAMELYON16)')
print(f'{"Model":<42} {"Val":>7} {"Test":>7} {"F1":>7} {"Gap":>8}')
print('-'*68)

for exp_id in ['E1_classical', 'E2_quantum', 'E4_resnet']:
    r = load_result(exp_id)
    if r:
        results[exp_id] = r
        print(f'{exp_files[exp_id]:<42} {r["val_auc"]:>7.4f} '
              f'{r["test_auc"]:>7.4f} {r["f1"]:>7.4f} {r["gap"]:>+8.4f}')
    else:
        print(f'{exp_files[exp_id]:<42} {"pending":>7}')

print()
print('TABLE 2 \u2014 Low-data regime (20% training data)')
print(f'{"Model":<42} {"Val":>7} {"Test":>7} {"F1":>7} {"Gap":>8}')
print('-'*68)

for exp_id in ['E3a_classical_20pct', 'E3b_quantum_20pct']:
    r = load_result(exp_id)
    if r:
        results[exp_id] = r
        print(f'{exp_files[exp_id]:<42} {r["val_auc"]:>7.4f} '
              f'{r["test_auc"]:>7.4f} {r["f1"]:>7.4f} {r["gap"]:>+8.4f}')
    else:
        print(f'{exp_files[exp_id]:<42} {"pending":>7}')

if 'E3a_classical_20pct' in results and 'E3b_quantum_20pct' in results:
    delta = results['E3b_quantum_20pct']['test_auc'] - results['E3a_classical_20pct']['test_auc']
    print(f'\nQuantum advantage at 20% data: {delta:+.4f} AUC', end='')
    print(' \u2705 CONFIRMED' if delta > 0 else ' \u26a0 not significant')

if 'E1_classical' in results and 'E4_resnet' in results:
    delta = results['E1_classical']['test_auc'] - results['E4_resnet']['test_auc']
    print(f'UNI contribution vs ResNet-50: {delta:+.4f} AUC')

print()
print('XAI figures: outputs/xai/')
print('Paper Figure 3: outputs/week6_xai_report.png')